[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/03_sanity_checks_and_gradcam.ipynb)

# R2-Q2 Week 3 — Sanity-check Grad-CAM, then run it on the misclassification set

This notebook does two things in order. First, it runs Adebayo et al. (2018)'s sanity checks on vanilla Grad-CAM applied to the R2-Q1 classifier — the model-parameter randomization check and the data-randomization check — and reports the Spearman rank correlation between trained-model and randomized-model heatmaps at each randomization stage, with ρ < 0.3 at full randomization committed as the passing criterion. Second, once vanilla Grad-CAM has passed the sanity checks, it generates heatmaps for every image in the working set produced by Week 2 and saves them for Week 4's categorization work.

By the end of this notebook you will have:

- A `sanity_check_results.json` recording the Spearman ρ trajectory across randomization stages for both the model-parameter and data-randomization checks, the verdict against the ρ < 0.3 passing criterion, and the methods-section text justifying that the trained model's downstream interpretation rests on a sanity-checked saliency method.
- A `heatmaps/` directory containing one `.npy` file per misclassification — 81 files, each a 7×7 numpy array (pre-upsampling) — plus a `gradcam_metadata.parquet` index that joins the heatmap files back to the working set on `filename`. Week 4 upsamples each heatmap at use time.
- A `shuffled_resnet18.pt` checkpoint — the label-shuffled-trained classifier used for the data-randomization sanity check, saved so the check is reproducible without retraining.
- A diagnostic figure (`sanity_check_trajectory.png`) showing the Spearman ρ curve across randomization stages, so a reader can see the trajectory the verdict was made from, not just the verdict itself.

## Before you run anything: switch to a GPU runtime

This notebook trains a neural network, which needs a more powerful processing 
unit (GPU) to work. By default, Colab gives you a CPU-only runtime — fine for 
last week's notebooks, but it won't work for this one.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive, set up irilab2026 with a GPU requirement, seed everything,
# and point OUTPUT_DIR at the R2-Q1 output directory.
iri.setup(gpu_required=True)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load the Week 1 pre-commit

This notebook does not read any specific field from `precommit.json`,
but the file is N01's signed deliverable, and downstream sanity-check
interpretation and Week-4 categorization both depend on it conceptually.
Loading it here is a guardrail: if the file is missing or malformed,
the cell below fails with a clear pointer back to N01 rather than
letting the notebook proceed into Grad-CAM work it can't actually
finish.

The load also light-validates the precommit's top-level structure —
the four keys `taxonomy`, `categorization_procedure`, `sanity_checks`,
and `aggregation_level` must all be present. If any are missing, the
precommit was probably written by an incomplete N01 run, and re-running
N01 to completion is the remedy.

In [ ]:
### 2.1) Load and validate the pre-commit

import json

precommit_path = OUTPUT_DIR / "precommit.json"

try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {precommit_path}. "
        "This file is produced by 01_pre_commits.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light validation: the four locked top-level keys must all be present.
expected_keys = {
    "taxonomy",
    "categorization_procedure",
    "sanity_checks",
    "aggregation_level",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "This usually means N01 didn't complete — "
        "re-run 01_pre_commits.ipynb to regenerate the file."
    )

print(f"Loaded: {precommit_path}")
print(f"  top-level keys: {sorted(precommit.keys())}")

### 2.2) Load the working set

N02 wrote `working_set.parquet` to this question's output directory: the
rows of the PlantDoc prediction table where the model misclassified at
the category level. This is the working set every Grad-CAM pass in
Sections 3, 4, and 5 runs over.

The schema validation below is minimal — it confirms that the two columns
this notebook actually uses are present: `filename` (the per-image
identifier for the heatmap files Section 5 writes) and `pred_idx` (the
Grad-CAM target class for every pass). A missing column means N02 wrote
an older schema; re-running N02 is the remedy.

In [ ]:
### 2.2) Load the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "working_set.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light schema validation: the columns this notebook actually uses must
# be present. `filename` is the per-image identifier that joins heatmap
# files back to working-set rows in Section 5; `pred_idx` is the Grad-CAM
# target class throughout Sections 3–5.
required_columns = {"filename", "pred_idx"}
missing = required_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"working_set.parquet is missing required columns: {sorted(missing)}. "
        "This usually means N02 wrote an older or partial schema — "
        "re-run 02_load_and_filter.ipynb to regenerate the file."
    )

# An empty working set means there are no misclassifications to inspect —
# methodologically interesting but operationally a stop condition for N03.
if len(working_set) == 0:
    raise ValueError(
        "working_set.parquet has zero rows. "
        "There are no misclassifications to inspect — "
        "if this is unexpected, check N02's filter logic."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows: {len(working_set)}")
print(f"  columns: {sorted(working_set.columns)}")

### 2.3) Load the classifier and check it matches the working set

R2-Q1 produced the trained classifier this question runs on top of. It
lives in R2-Q1's output directory, not R2-Q2's — the cell below points
at it explicitly and loads it.

Two non-obvious choices in the code below:

First, `num_classes` is derived from the saved state_dict's `fc.weight`
shape, not hardcoded as 38 or read from a precommit. This keeps N03
decoupled from how R2-Q1 happened to number its classes — if a future
R2-Q1 run trains against a different class count, this notebook still
loads the model correctly.

Second, after loading, the cell cross-checks that the working set's max
`pred_idx` is less than the model's `num_classes`. If it isn't, the
working set was built against a different model than the one just loaded,
and any Grad-CAM target class index downstream would be meaningless.
Catching this here is far less painful than chasing a silent
miscategorization in Week 4.

The cell finishes with a forward pass on a dummy input — a small
assurance that the model loaded structurally cleanly and produces logits
of the expected shape.

In [ ]:
### 2.3) Load the classifier and check it matches the working set

import torch

# R2-Q1's output directory holds the trained classifier this notebook inherits.
R2Q1_DIR = iri.output_dir("r2_q1")
classifier_path = R2Q1_DIR / "baseline_resnet18.pt"

try:
    state_dict = torch.load(classifier_path, weights_only=True, map_location="cpu")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {classifier_path}. "
        "This file is produced by R2-Q1's 03_baseline_classifier.ipynb — "
        "run that notebook in your R2-Q1 series first, "
        "or ensure your R2-Q1 output directory is mounted."
    ) from None

# Derive num_classes from the saved head's output shape. ResNet-18's
# fc.weight is (num_classes, 512), so its first dimension is the class
# count the saved model was trained against.
num_classes = state_dict["fc.weight"].shape[0]
print(f"num_classes (from state_dict fc.weight): {num_classes}")

# Build the matching architecture and load the trained weights.
# pretrained=False because we're loading trained weights anyway — no
# point downloading the ImageNet head we're about to overwrite.
model = iri.build_baseline_model(num_classes, pretrained=False)
model.load_state_dict(state_dict)
model = model.cuda().eval()

# Cross-check: every working-set row has a pred_idx that must be a valid
# class index for this model. If the max pred_idx is not less than
# num_classes, the working set and classifier were produced from
# different configurations and one of them needs to be regenerated.
max_pred_idx = int(working_set["pred_idx"].max())
assert max_pred_idx < num_classes, (
    f"working set pred_idx max ({max_pred_idx}) is not less than "
    f"the loaded model's num_classes ({num_classes}). "
    "The working set and classifier were produced from different "
    "configurations; one of them needs to be regenerated."
)
print(f"pred_idx max ({max_pred_idx}) < num_classes ({num_classes}) — OK")

# Smoke check: a forward pass on a dummy input confirms the loaded model
# is structurally sound and produces logits of the expected shape.
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device="cuda")
    out = model(dummy)
assert out.shape == (1, num_classes), (
    f"forward pass produced output shape {tuple(out.shape)}, "
    f"expected (1, {num_classes})."
)
print(f"smoke-check forward pass output shape: {tuple(out.shape)} — OK")

## 3) Adebayo sanity check 1 — model-parameter randomization

Grad-CAM produces a heatmap by combining a convolutional layer's
activations with the gradient of the predicted class score with
respect to those activations. The heatmap is a story about *where*
the model looked; this section answers whether that story is
actually a function of what the model learned.

Adebayo et al. (2018) test this directly: take the trained model,
progressively randomize its parameters layer by layer (from logits
down to layer1), and recompute the heatmap at each stage. If the
heatmap depends on the model's learning, randomization should
disrupt it — measurable as a falling rank correlation between the
trained-model heatmap and the randomized one. If the heatmap
*doesn't* change much, it's a function of the input only, and
tells you nothing about the trained model — meaning any failure-mode
interpretation built on top of it in Week 4 would be unsupported.

The passing criterion you committed to in N01: Spearman ρ < 0.3
at full randomization.

This section has five subsections:

- **3.1** Define Grad-CAM. Confirm the implementation works on a
  handful of working-set images before depending on it for the
  formal check.
- **3.2** Pick a 12-image probe set from the working set, with a
  fixed seed.
- **3.3** Compute trained-model heatmaps for the probe set.
- **3.4** Randomize the model layer by layer, recompute heatmaps
  at each stage, compute Spearman ρ against the trained-model
  heatmaps.
- **3.5** Summarize the ρ trajectory, plot it, and evaluate the
  verdict against the ρ < 0.3 criterion.

### 3.1) Grad-CAM mechanics and implementation validity

Grad-CAM (Selvaraju et al. 2017) localizes class-discriminative
evidence in an image. Given a trained model, an input image, and
a target class:

1. **Forward-pass** the image through the model and capture the
   activations of a chosen convolutional layer. For ResNet-18 the
   standard choice is `model.layer4`, the last conv block — its
   output is a 7×7 feature map with 512 channels.
2. **Backward-pass** from the target-class logit and capture the
   gradient of that logit with respect to those activations.
3. **Globally average-pool** the gradients over the spatial
   dimensions to get one weight per channel — these weights are
   how much each channel mattered for the target-class score.
4. **Weighted sum** of activation channels, then **ReLU**. The
   result is a 7×7 heatmap with non-negative values, high where
   activations contributed positively to the target-class score.

The first code cell below defines the machinery: a
`load_pd_image(filename)` helper that loads a single PlantDoc image
by its filename and applies the eval transform, and a
`compute_gradcam(model, image_tensor, target_class, target_layer)`
function that implements the four steps above. Both functions get
reused throughout Sections 3, 4, and 5.

The second code cell runs the implementation-validity check: load
the first three working-set images, compute their heatmaps using
the wrongly-predicted class (`pred_idx`) as the target, and plot
the result. If the implementation has a bug — hooks not firing,
gradients not flowing, wrong target layer, autograd disabled
somewhere — this catches it now, before the formal Adebayo check
in 3.3–3.4 starts producing meaningless ρ values. We're not
asking whether the heatmap is *right* yet (Week 4 does that); just
whether it's a non-degenerate function of the image and model.

In [ ]:
### 3.1) Define the machinery — PD image loader and compute_gradcam

import torch.nn.functional as F

# Load PlantDoc metadata and image data. First call downloads ~950 MB
# from Hugging Face and caches it to Drive; subsequent calls hit the
# cache. The metadata DataFrame has one row per PD image with a
# `filename` column that matches the working set's `filename` join key.
pd_metadata, pd_hf_dataset = iri.load_plantdoc()
print(f"PlantDoc loaded: {len(pd_metadata)} rows")

# Build a filename → pandas-index lookup so we can pull a specific
# image by its filename in O(1). The HF dataset is index-aligned to
# the metadata, so the pandas index doubles as the hf_dataset position.
filename_to_pd_index = dict(zip(pd_metadata["filename"], pd_metadata.index))

# Eval transform: Resize(256) → CenterCrop(224) → ToTensor → Normalize.
# Same transform R2-Q1 used at evaluation time, so the pixels this
# notebook feeds the model match what R2-Q1's predictions were
# computed on.
eval_tf = iri.imagenet_eval_transform()


def load_pd_image(filename):
    """Load a PlantDoc image by filename and apply the eval transform.

    Parameters
    ----------
    filename : str
        The PD filename, as stored in `working_set["filename"]`.

    Returns
    -------
    image_tensor : torch.Tensor
        Shape (3, 224, 224), ImageNet-normalized. The caller is
        responsible for `unsqueeze(0)` and `.cuda()` before feeding
        the model — `compute_gradcam` below does both internally.
    """
    if filename not in filename_to_pd_index:
        raise KeyError(
            f"PlantDoc filename {filename!r} not found in metadata. "
            "This shouldn't happen — working-set filenames came from "
            "pd_predictions.parquet, built against this same PD dataset."
        )
    pd_index = filename_to_pd_index[filename]
    image = pd_hf_dataset[int(pd_index)]["image"].convert("RGB")
    return eval_tf(image)


def compute_gradcam(model, image_tensor, target_class, target_layer):
    """Compute a vanilla Grad-CAM heatmap (Selvaraju et al. 2017).

    Forward-passes the image, captures activations at `target_layer`,
    backward-passes the target-class logit, captures gradients at
    `target_layer`, globally average-pools the gradients to get
    per-channel weights, takes the weighted sum of activations, ReLU.

    Parameters
    ----------
    model : nn.Module
        Trained classifier, in eval mode, on CUDA.
    image_tensor : torch.Tensor
        Shape (3, 224, 224), ImageNet-normalized. Unsqueezed and
        moved to CUDA inside the function.
    target_class : int
        Class index to compute the heatmap for. For working-set
        misclassifications, this is `pred_idx` — the class the model
        wrongly predicted.
    target_layer : nn.Module
        The conv layer to hook. Typically `model.layer4` for
        ResNet-18 (the last conv block, 7×7×512).

    Returns
    -------
    heatmap : np.ndarray
        Shape (7, 7) for ResNet-18 layer4, non-negative float32.
        Higher values where activations contributed more to the
        target-class score. Pre-upsampling — N04 upsamples to the
        image's native resolution at use time.
    """
    activations = []
    gradients = []

    def fwd_hook(module, input, output):
        activations.append(output)

    def bwd_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    fwd_handle = target_layer.register_forward_hook(fwd_hook)
    bwd_handle = target_layer.register_full_backward_hook(bwd_hook)

    try:
        model.zero_grad()
        x = image_tensor.unsqueeze(0).cuda()           # (1, 3, 224, 224)
        logits = model(x)                              # (1, num_classes)
        logits[0, target_class].backward()

        A = activations[0]                             # (1, C, H, W)
        dA = gradients[0]                              # (1, C, H, W)
        weights = dA.mean(dim=(2, 3), keepdim=True)    # (1, C, 1, 1)
        cam = (weights * A).sum(dim=1)                 # (1, H, W)
        cam = F.relu(cam).squeeze(0)                   # (H, W)
        cam = cam.detach().cpu().numpy()
    finally:
        fwd_handle.remove()
        bwd_handle.remove()

    return cam


print("Defined: load_pd_image, compute_gradcam")

### 3.2) Pick the probe set

Adebayo's original paper ran the sanity check on a single image.
Aggregating across a small probe set gives a more defensible
characterization of average behavior — the mean ρ across multiple
images is less sensitive to whether one image happens to anchor
the check. We pick 12 images, large enough that the mean is
stable but small enough that the full randomization sweep
(six stages × twelve images = 72 Grad-CAM passes) runs in seconds.

The sample is drawn with a fixed seed so re-running this notebook
produces the same probe set. The same probe set carries through
3.3 (compute trained-model heatmaps), 3.4 (recompute at each
randomization stage), 3.5 (summarize trajectory), and Section 4
(data-randomization check). Pre-computing the input tensors here
means every downstream cell iterates over a list of ready-to-go
tensors rather than reloading from the HF dataset each time.

In [ ]:
### 3.2) Pick the probe set

PROBE_SEED = 42
PROBE_SIZE = 12

# Fixed-seed sample from the working set. random_state=PROBE_SEED makes
# this deterministic across sessions: re-running gives the same 12 rows.
# reset_index(drop=True) gives clean 0..N-1 positional indices for the
# downstream loops in 3.3, 3.4, and Section 4.
probe_set = working_set.sample(
    n=PROBE_SIZE, random_state=PROBE_SEED
).reset_index(drop=True)

# Pre-compute the input tensors so the randomization sweep in 3.4 can
# iterate without reloading from the HF dataset on each pass. Twelve
# tensors × 3 × 224 × 224 × 4 bytes ≈ 7 MB — trivial; held in CPU
# memory and moved to GPU inside compute_gradcam at call time.
probe_tensors = [load_pd_image(fn) for fn in probe_set["filename"]]

print(
    f"Probe set: {len(probe_set)} of {len(working_set)} working-set rows "
    f"(seed={PROBE_SEED})"
)
print(f"  true categories: {probe_set['true_category'].value_counts().to_dict()}")
print(f"  pred categories: {probe_set['pred_category'].value_counts().to_dict()}")

### 3.3) Compute trained-model heatmaps for the probe set

The trained-model heatmaps are the baseline 3.4's randomized-model
heatmaps get compared against. Two choices fixed in 3.1 carry
forward here and through 3.4:

- **Target class** is the trained model's predicted class
  (`pred_idx`), held constant across all randomization stages.
  This is the Adebayo adaptation — the comparison in 3.4 is "does
  the heatmap for the *same target* change when the model
  changes?" Letting the target drift would conflate model effects
  with target effects.
- **Target layer** is `model.layer4`, the standard ResNet-18
  choice. Hardcoded at the call site so the choice is visible.

The cell stacks the 12 heatmaps into a `(12, 7, 7)` array. 3.4
produces a parallel array at each randomization stage and computes
Spearman ρ against this baseline, row by row.

In [ ]:
### 3.3) Compute trained-model heatmaps for the probe set

import numpy as np

# Trained-model heatmaps — the baseline 3.4's randomized heatmaps
# get compared against. Per Adebayo's adaptation, target_class is
# the trained model's predicted class (pred_idx) and stays constant
# across all randomization stages in 3.4.
trained_heatmaps = np.stack([
    compute_gradcam(model, tensor, int(pred_idx), model.layer4)
    for tensor, pred_idx in zip(probe_tensors, probe_set["pred_idx"])
])

print(f"trained_heatmaps shape: {trained_heatmaps.shape}")
print(f"  value range: [{trained_heatmaps.min():.3f}, {trained_heatmaps.max():.3f}]")
print(f"  mean per-image std: {trained_heatmaps.std(axis=(1, 2)).mean():.3f}")

### 3.4) Randomize layer-by-layer, recompute heatmaps, compute Spearman ρ

Six stages, top-down. At each stage past the first, the listed
layers are re-initialized using their default initializers
(Kaiming for convs and linears, ones/zeros for batch-norm affine
parameters; batch-norm running stats reset). The stages are
cumulative — by the last stage, every layer with parameters has
been randomized.

| Stage     | Randomized layers                                  |
|-----------|----------------------------------------------------|
| untouched | (none — trained model)                             |
| logits    | fc                                                 |
| layer4    | fc, layer4                                         |
| layer3    | fc, layer4, layer3                                 |
| layer2    | fc, layer4, layer3, layer2                         |
| layer1    | fc, layer4, layer3, layer2, layer1, conv1, bn1     |

At each stage we recompute Grad-CAM on the same 12 probe images
with the same target class (the trained model's `pred_idx`), then
compute Spearman ρ per probe image against the trained-model
heatmaps from 3.3. The ρ values land in a long-format trajectory
dataframe (`probe_idx`, `stage`, `rho`) that 3.5 summarizes and
plots.

The randomization runs on a separate `rand_model` instance so the
original `model` stays loaded with trained weights for Section 5's
full working-set pass. Each stage reloads the trained state into
`rand_model` from a snapshot, then re-randomizes the cumulative
layer set.

In [ ]:
### 3.4) Randomize layer-by-layer, recompute heatmaps, compute Spearman ρ

from scipy.stats import spearmanr

# Adebayo's cascading stages — top-down, cumulative.
# Stage 0 ("untouched") is the trained model itself; ρ should be ~1.0
# trivially (sanity check at the end of the cell catches a regression).
STAGES = [
    ("untouched", []),
    ("logits",    ["fc"]),
    ("layer4",    ["fc", "layer4"]),
    ("layer3",    ["fc", "layer4", "layer3"]),
    ("layer2",    ["fc", "layer4", "layer3", "layer2"]),
    ("layer1",    ["fc", "layer4", "layer3", "layer2", "layer1", "conv1", "bn1"]),
]


def randomize_module(module):
    """Re-initialize parameters of all submodules under `module`.

    Walks recursively; any submodule with reset_parameters() (Conv2d,
    BatchNorm2d, Linear) gets re-initialized with its default
    initializer. For BatchNorm2d, reset_parameters() also resets the
    running statistics.
    """
    for m in module.modules():
        if hasattr(m, "reset_parameters"):
            m.reset_parameters()


# Snapshot the trained weights once. load_state_dict copies values into
# the target model's parameters, so trained_state stays untouched even
# as we randomize rand_model's params in the loop.
trained_state = model.state_dict()

# Separate model for randomization. Keeps the original `model` clean for
# Section 5's full working-set pass which needs the trained weights.
rand_model = iri.build_baseline_model(num_classes, pretrained=False).cuda().eval()

# Seed the randomization sweep for reproducibility — every run produces
# the same trajectory dataframe given the same trained_state and probe set.
iri.seed_all(42)

trajectory_rows = []
for stage_name, layers_to_randomize in STAGES:
    # Refresh from trained weights, then randomize the listed layers.
    rand_model.load_state_dict(trained_state)
    for layer_name in layers_to_randomize:
        randomize_module(getattr(rand_model, layer_name))

    # Heatmaps with the (possibly-randomized) rand_model. Target class
    # is the trained model's pred_idx, held constant across stages per
    # Adebayo's adaptation.
    stage_heatmaps = np.stack([
        compute_gradcam(rand_model, tensor, int(pred_idx), rand_model.layer4)
        for tensor, pred_idx in zip(probe_tensors, probe_set["pred_idx"])
    ])

    # Spearman ρ per probe image between trained and randomized heatmaps.
    # Flatten the 7×7 grids before correlating — Spearman is on ranks of
    # all 49 spatial positions.
    #
    # Randomized models can produce all-zero heatmaps when the random
    # weights push the weighted sum non-positive everywhere before the
    # ReLU. spearmanr is undefined on a constant input, so we treat that
    # case as ρ = 0: the randomized model produced no discriminative
    # attention to correlate with the trained baseline. is_degenerate
    # preserves the diagnostic for 3.5 (the verdict isn't sensitive to
    # how many probes were degenerate — degenerate probes contribute 0
    # to the mean, pulling it down — but the count is worth surfacing).
    stage_rhos = []
    for probe_idx in range(PROBE_SIZE):
        stage_flat = stage_heatmaps[probe_idx].flatten()
        if stage_flat.std() < 1e-9:
            rho = 0.0
            is_degenerate = True
        else:
            rho, _ = spearmanr(
                trained_heatmaps[probe_idx].flatten(),
                stage_flat,
            )
            is_degenerate = False

        stage_rhos.append(rho)
        trajectory_rows.append({
            "probe_idx": probe_idx,
            "stage": stage_name,
            "rho": rho,
            "is_degenerate": is_degenerate,
        })

    n_deg = sum(1 for r in trajectory_rows[-PROBE_SIZE:] if r["is_degenerate"])
    deg_note = f"  ({n_deg}/{PROBE_SIZE} degenerate)" if n_deg else ""
    print(f"  {stage_name:12s}: mean ρ = {np.mean(stage_rhos):.3f}{deg_note}")

trajectory = pd.DataFrame(trajectory_rows)
print(f"\nTrajectory dataframe: {len(trajectory)} rows "
      f"({PROBE_SIZE} probes × {len(STAGES)} stages)")

# Sanity: the untouched stage should give ρ ≈ 1.0 (rand_model has
# trained weights at this stage, so its heatmaps should match the
# trained model's). If this fires, the state_dict load isn't applying
# correctly.
assert trajectory.query("stage == 'untouched'")["rho"].mean() > 0.99, (
    "Untouched-stage mean ρ should be ~1.0 — rand_model has the trained "
    "weights loaded at this stage, so its heatmaps should match the "
    "trained-model baseline. Got a lower value; something's wrong with "
    "the state_dict load or the model build."
)

### 3.5) Summarize the trajectory, plot it, evaluate the verdict

This cell does three things in order:

1. **Per-stage summary.** Mean ρ across the probe set, standard
   deviation, and degenerate count at each stage. Six rows — the
   bird's-eye view of what 3.4 produced.
2. **Plot the trajectory.** Saved to `sanity_check_trajectory.png`
   in the output directory. Shows the mean ρ line, per-probe
   scatter (degenerate probes marked with ×), and the ±0.3
   criterion bands. A reader can see the verdict trajectory at a
   glance, not just the final number.
3. **Evaluate the verdict.** Per the precommit, ρ < 0.3 at full
   randomization. We read that as the saliency-literature
   absolute-value convention `|ρ| < 0.3` — strong negative
   correlation also fails to "preserve" the trained model's
   attention. The verdict comes from the mean ρ at the last stage
   (`layer1`), the point at which every learnable parameter has
   been randomized.

If the verdict fails, the cell prints a warning that downstream
work in Section 5 is methodologically compromised. The notebook
doesn't hard-stop — the student sees the warning and decides
whether to continue.

In [ ]:
### 3.5) Summarize the trajectory, plot it, evaluate the verdict
import matplotlib.pyplot as plt
# Per-stage summary across the probe set.
stage_summary = (
    trajectory
    .groupby("stage", sort=False)
    .agg(
        mean_rho=("rho", "mean"),
        std_rho=("rho", "std"),
        n_degenerate=("is_degenerate", "sum"),
    )
    .reset_index()
)
print("Per-stage summary:")
print(stage_summary.to_string(index=False))
print()

# Plot the trajectory: mean ρ line + per-probe scatter + criterion bands.
stage_order = [name for name, _ in STAGES]
stage_x = list(range(len(stage_order)))

fig, ax = plt.subplots(figsize=(8, 5))

# Per-probe scatter with horizontal jitter so 12 points at the same
# stage don't pile up at one x value. Seeded RNG keeps the jitter
# reproducible across runs.
rng = np.random.default_rng(0)
for stage_i, stage_name in enumerate(stage_order):
    stage_rows = trajectory[trajectory["stage"] == stage_name].reset_index(drop=True)
    is_deg = stage_rows["is_degenerate"].values
    x_jitter = stage_i + rng.uniform(-0.1, 0.1, size=len(stage_rows))

    ax.scatter(x_jitter[~is_deg], stage_rows.loc[~is_deg, "rho"],
               alpha=0.5, color="steelblue", s=40)
    if is_deg.any():
        ax.scatter(x_jitter[is_deg], stage_rows.loc[is_deg, "rho"],
                   alpha=0.7, marker="x", color="firebrick", s=50)

# Mean ρ line on top of the scatter.
ax.plot(stage_x, stage_summary["mean_rho"], color="black", linewidth=2,
        marker="o", markersize=8, zorder=3)

# Criterion bands at ±0.3 (the absolute-value reading of the precommit).
ax.axhline(0.3, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(-0.3, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.axhspan(-0.3, 0.3, alpha=0.05, color="gray")

# Legend handles via empty-scatter trick — keeps the three categories
# in a fixed order regardless of which appear in the data.
ax.scatter([], [], color="steelblue", alpha=0.5, s=40, label="probe ρ")
ax.scatter([], [], color="firebrick", marker="x", alpha=0.7, s=50, label="degenerate probe")
ax.plot([], [], color="black", linewidth=2, marker="o", markersize=8, label="mean ρ")

ax.set_xticks(stage_x)
ax.set_xticklabels(stage_order)
ax.set_xlabel("Randomization stage (cumulative, top-down)")
ax.set_ylabel("Spearman ρ vs trained-model heatmap")
ax.set_title("Adebayo model-parameter randomization trajectory")
ax.set_ylim(-1.05, 1.05)
ax.legend(loc="best")
ax.grid(True, alpha=0.3)

fig.tight_layout()

trajectory_path = OUTPUT_DIR / "sanity_check_trajectory.png"
fig.savefig(trajectory_path, dpi=150)
plt.show()
print(f"Plot saved: {trajectory_path}")
print()

# Verdict at full randomization (last stage). |mean ρ| < 0.3 is the
# absolute-value reading; strong negative correlation would also fail
# to "preserve" the trained model's attention.
full_random_mean_rho = stage_summary.loc[
    stage_summary["stage"] == "layer1", "mean_rho"
].iloc[0]
param_random_passes = abs(full_random_mean_rho) < 0.3

if param_random_passes:
    print("VERDICT: PASS")
    print(f"  |mean ρ at full randomization| = {abs(full_random_mean_rho):.3f} < 0.3")
    print("  Grad-CAM passes Adebayo's model-parameter randomization check.")
else:
    print("VERDICT: FAIL")
    print(f"  |mean ρ at full randomization| = {abs(full_random_mean_rho):.3f} >= 0.3")
    print()
    print("  WARNING: Grad-CAM does not pass Adebayo's model-parameter")
    print("  randomization check on this trained model. The heatmaps in")
    print("  Section 5 would not be certifiably a function of what the")
    print("  model learned. Any failure-mode interpretation built on top")
    print("  of them in Week 4 would be methodologically unsupported.")

## 4) Adebayo sanity check 2 — data randomization

Section 3 randomized the model parameters; Section 4 randomizes
the training data. Take the same training recipe, shuffle the
labels across all training images, retrain from scratch. The
resulting model has been "trained" on noise — there's no learnable
relationship between input image and (shuffled) label. If
Grad-CAM heatmaps from the shuffled-label model still resemble
those from the genuinely-trained model, the heatmaps are a
function of input-data structure (e.g., where the leaf sits in
the frame), not of what the model learned.

The criterion, same shape as Section 3: `|mean ρ| < 0.3` across
the 12-probe set at the data-randomization stage.

This section has two subsections:

- **4.1** Train (or load cached) the shuffled-label model. The
  retrain runs R2-Q1's full recipe with `shuffle_labels=True` on
  PlantVillage; takes ~30 minutes on a T4. The cell caches the
  result to disk so re-runs skip the training pass.
- **4.2** Compute shuffled-model heatmaps on the existing
  12-probe set (the same probes Section 3 used) with the same
  target classes (`pred_idx` from the trained model). Spearman
  ρ per probe, mean across probes, verdict against the
  `|ρ| < 0.3` criterion.

### 4.1) Train (or load cached) the shuffled-label model

`iri.train_baseline` runs R2-Q1's training recipe (ResNet-18, SGD
with momentum, lr 0.01 → 0.001 at epoch 7, 10 epochs, batch size
32, 10% stratified val carve). With `shuffle_labels=True`, class
indices are shuffled across the train split before the val carve
— both train and val carry the shuffled labels, and the model
fits to noise. Val accuracy should hover near chance
(`1/num_classes`) for all epochs.

`DEV_EPOCH_CAP` is the dev-iteration knob: set it to a small
integer (e.g., 2) to cap epochs and iterate quickly during
development; leave it as `None` for the full 10-epoch recipe.
**The verdict cell in 4.2 refuses to mark `pass` when
`DEV_EPOCH_CAP` is in effect** — a truncated retrain doesn't meet
the recipe and any "pass" would be methodologically untrustworthy.

The cell checks for `shuffled_resnet18.pt` in the output directory
first. If present, the file is loaded and the training pass is
skipped; if absent, training runs (~30 minutes on a T4 at full
recipe) and the result is saved. Re-running 4.1 after a successful
training pass is near-instant.

In [ ]:
### 4.1) Train (or load cached) the shuffled-label model

DEV_EPOCH_CAP = None   # set to a small int (e.g., 2) for dev iteration

shuffled_path = OUTPUT_DIR / "shuffled_resnet18.pt"

if shuffled_path.exists():
    # Cached retrain from a previous run — load and skip training.
    print(f"Loading existing shuffled model: {shuffled_path}")
    checkpoint = torch.load(
        shuffled_path, weights_only=True, map_location="cpu"
    )
    # New format is a dict {state_dict, epoch_cap}; bare state_dicts from
    # before this fix get the conservative treatment (treated as truncated).
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        shuffled_state = checkpoint["state_dict"]
        cached_epoch_cap = checkpoint.get("epoch_cap", "unknown")
    else:
        shuffled_state = checkpoint
        cached_epoch_cap = "unknown"
    history_shuffled = None  # history isn't needed downstream, so we don't have to reconstruct it from the cache
else:
    # No cached file — train from scratch via the library helper.
    print("No cached shuffled model found. Training from scratch...")
    print("(~30 minutes on a T4 at full recipe; less if DEV_EPOCH_CAP is set)\n")

    pv_metadata, pv_hf_dataset = iri.load_plantvillage()
    shuffled_state, history_shuffled = iri.train_baseline(
        pv_metadata, pv_hf_dataset,
        dataset_class=iri.PlantVillageDataset,
        num_classes=num_classes,
        shuffle_labels=True,
        seed=42,
        epoch_cap=DEV_EPOCH_CAP,
        verbose=True,
    )
    # Wrap the state dict with provenance so the verdict cell in 4.2 can tell
    # whether this checkpoint was trained to the full recipe or truncated.
    torch.save(
        {
            "state_dict": shuffled_state,
            "epoch_cap": DEV_EPOCH_CAP,  # None = full recipe; int = truncated
        },
        shuffled_path,
    )
    cached_epoch_cap = DEV_EPOCH_CAP   # ← ADD THIS LINE
    print(f"\nSaved: {shuffled_path}")
    print(
        f"  best_val_acc = {history_shuffled['best_val_acc']:.4f}  "
        f"(expected near 1/{num_classes} ≈ {1/num_classes:.3f})"
    )

# Build the matching architecture, load the shuffled weights, move to GPU.
# Same pattern as Section 2.3's load of the trained classifier.
shuffled_model = iri.build_baseline_model(
    num_classes, pretrained=False
).cuda().eval()
shuffled_model.load_state_dict(shuffled_state)
print("\nShuffled model ready.")

### 4.2) Compute shuffled-model heatmaps and evaluate the verdict

Grad-CAM on the same 12-probe set used in Section 3, with the same
target classes (the trained model's `pred_idx`, held constant
across the data-randomization comparison just as it was across
Section 3's model-parameter stages). Spearman ρ per probe, mean
across probes, verdict against the `|ρ| < 0.3` criterion.

Two structural notes carried over from Section 3:

- **Degenerate heatmaps.** The shuffled-label model can produce
  all-zero heatmaps when the trained-on-noise weights happen to
  push the layer4 weighted sum non-positive everywhere before
  the ReLU. Same handling as 3.4 — ρ = 0 for those probes,
  flagged in the per-probe breakdown.
- **`DEV_EPOCH_CAP` refusal.** If 4.1 was run with a truncated
  epoch count, the verdict is withheld regardless of the
  computed mean ρ.

In [ ]:
### 4.2) Compute shuffled-model heatmaps, evaluate the verdict

# Same probe set, same target classes (pred_idx — the trained model's
# predicted classes), now with the shuffled-label model.
shuffled_heatmaps = np.stack([
    compute_gradcam(shuffled_model, tensor, int(pred_idx), shuffled_model.layer4)
    for tensor, pred_idx in zip(probe_tensors, probe_set["pred_idx"])
])

# Spearman ρ per probe image. Same degeneracy handling as Section 3.4:
# if the shuffled-model heatmap is constant (all-zero after ReLU), treat
# ρ as 0 — the shuffled model produced no discriminative attention to
# correlate with the trained baseline. Flat lists, not a dataframe —
# this is a single-stage comparison, not a trajectory.
data_rhos = []
data_degenerate = []
for probe_idx in range(PROBE_SIZE):
    shuffled_flat = shuffled_heatmaps[probe_idx].flatten()
    if shuffled_flat.std() < 1e-9:
        rho = 0.0
        is_degenerate = True
    else:
        rho, _ = spearmanr(
            trained_heatmaps[probe_idx].flatten(),
            shuffled_flat,
        )
        is_degenerate = False
    data_rhos.append(rho)
    data_degenerate.append(is_degenerate)

data_random_mean_rho = float(np.mean(data_rhos))
n_degenerate = sum(data_degenerate)

# Per-probe breakdown — same diagnostic role as Section 3.5's per-stage
# table, but per-probe since Section 4 is a single comparison.
print("Per-probe ρ vs trained-model heatmap:")
for probe_idx, (rho, deg) in enumerate(zip(data_rhos, data_degenerate)):
    deg_marker = "  (degenerate)" if deg else ""
    print(f"  probe {probe_idx:2d}: ρ = {rho:+.3f}{deg_marker}")
print()
print(
    f"Mean ρ across {PROBE_SIZE} probes: {data_random_mean_rho:+.3f}  "
    f"({n_degenerate}/{PROBE_SIZE} degenerate)"
)
print()

# Verdict. Same absolute-value reading as Section 3.5. If DEV_EPOCH_CAP
# was in effect during 4.1, the retrain is truncated and the verdict is
# withheld regardless of the computed mean ρ.
data_random_passes_raw = abs(data_random_mean_rho) < 0.3

# Withhold if EITHER the current run is truncated OR the loaded checkpoint
# was trained truncated. The cached-truncated case is the subtle one — a
# stale dev-mode checkpoint sitting on disk, loaded silently by Cell 4.1
# when the student re-runs with DEV_EPOCH_CAP = None.
run_truncated = DEV_EPOCH_CAP is not None
checkpoint_truncated = cached_epoch_cap is not None  # int or "unknown" → truncated

if run_truncated or checkpoint_truncated:
    data_random_passes = False
    data_random_verdict = "withheld"    # ← ADD
    reason_parts = []
    if run_truncated:
        reason_parts.append(f"this run has DEV_EPOCH_CAP = {DEV_EPOCH_CAP}")
    if checkpoint_truncated:
        if cached_epoch_cap == "unknown":
            reason_parts.append("cached checkpoint has no epoch provenance")
        else:
            reason_parts.append(f"cached checkpoint was trained with epoch_cap = {cached_epoch_cap}")
    print(f"VERDICT: WITHHELD ({'; '.join(reason_parts)})")
    print(f"  |mean ρ| = {abs(data_random_mean_rho):.3f}")
    print("  The shuffled retrain was truncated; the recipe wasn't run to")
    print("  completion. The verdict isn't valid until the full 10-epoch")
    print("  recipe runs. Set DEV_EPOCH_CAP = None and delete")
    print("  shuffled_resnet18.pt to force a fresh full retrain.")
elif data_random_passes_raw:
    data_random_passes = True
    data_random_verdict = "pass"
    print("VERDICT: PASS")
    print(f"  |mean ρ at data-randomization| = "
          f"{abs(data_random_mean_rho):.3f} < 0.3")
    print("  Grad-CAM passes Adebayo's data-randomization check.")
else:
    data_random_passes = False
    data_random_verdict = "fail"
    print("VERDICT: FAIL")
    print(f"  |mean ρ at data-randomization| = "
          f"{abs(data_random_mean_rho):.3f} >= 0.3")
    print()
    print("  WARNING: Grad-CAM does not pass Adebayo's data-randomization")
    print("  check. The trained-model heatmaps are too similar to those")
    print("  produced by a model trained on random labels — the saliency")
    print("  method may be picking up on input-data structure rather than")
    print("  features the model learned. Failure-mode interpretation in")
    print("  Week 4 would be unsupported.")

## 5) Generate Grad-CAM heatmaps for the full working set

With both sanity checks passed, Grad-CAM is certified for this
trained model — heatmaps are a function of what the model learned,
not of input-data structure or the saliency method's internals.
Section 5 produces the deliverable Week 4 consumes: one heatmap
per misclassification, saved per-image so Week 4 can load them
individually without holding all 81 in memory.

Two artifacts are produced:

- `heatmaps/` — a subdirectory in the output directory containing
  one `.npy` file per working-set row. Each file holds a 7×7
  float numpy array, the raw layer4 Grad-CAM heatmap
  pre-upsampling. Week 4 upsamples to the original image's
  resolution at use time, when it knows what resolution the
  per-image categorization predicates need.
- `gradcam_metadata.parquet` — the index that maps working-set
  rows to `.npy` paths. Three columns: `filename` (the join
  key to `working_set.parquet`), `heatmap_path` (relative path
  inside the output directory), `target_class` (the trained
  model's `pred_idx` — the Grad-CAM target class, recorded for
  the methods section).

The work splits into two cells: the heatmap-generation loop (the
81-image pass), and the metadata save plus round-trip verify.

### 5.1) Heatmap-generation loop

Same call pattern as Section 3.3's trained-model heatmaps: target
class is `pred_idx` (the wrongly-predicted class), target layer
is `model.layer4`. The difference is scope — Section 3.3 ran on
12 probe images for the sanity check, Section 5 runs on the full
81-row working set.

Each iteration loads the image via `load_pd_image`, computes the
Grad-CAM heatmap, saves the 7×7 numpy array to
`heatmaps/{filename_stem}.npy`, and appends a row to the
in-memory metadata list. Progress prints every 10 images. At
~10 ms per Grad-CAM pass on a T4, the full loop runs in well
under a minute including image-load overhead.

In [ ]:
### 5.1+5.2) Generate Grad-CAM heatmaps for the full working set

from pathlib import Path

# Set up the heatmaps subdirectory. exist_ok=True so re-running this
# section (which is fast — no reason not to) doesn't error if the
# directory is already there.
heatmaps_dir = OUTPUT_DIR / "heatmaps"
heatmaps_dir.mkdir(exist_ok=True)

# Build the .npy save path for each working-set row up-front.
# Path(filename).stem strips the image extension. PD filenames should
# all be unique; the assert below catches the case where two of them
# share a stem after extension stripping (would silently overwrite).
heatmap_paths = [
    heatmaps_dir / f"{Path(fn).stem}.npy"
    for fn in working_set["filename"]
]
assert len(set(heatmap_paths)) == len(heatmap_paths), (
    "Two working-set filenames share a stem after extension stripping. "
    "Heatmap save paths would collide and silently overwrite. "
    "If this fires, switch to indexed filenames (heatmap_000.npy, ...) "
    "and store the original filename in the metadata column."
)

# Main loop: compute Grad-CAM, save .npy, accumulate metadata row.
metadata_rows = []
for i, (_, row) in enumerate(working_set.iterrows()):
    image_tensor = load_pd_image(row["filename"])
    heatmap = compute_gradcam(
        model, image_tensor, int(row["pred_idx"]), model.layer4
    )

    heatmap_path = heatmap_paths[i]
    np.save(heatmap_path, heatmap)

    metadata_rows.append({
        "filename": row["filename"],
        "heatmap_path": str(heatmap_path.relative_to(OUTPUT_DIR)),
        "target_class": int(row["pred_idx"]),
    })

    if (i + 1) % 10 == 0 or i == len(working_set) - 1:
        print(f"  {i + 1:3d}/{len(working_set)} heatmaps computed")

gradcam_metadata = pd.DataFrame(metadata_rows)
print(f"\nWrote {len(gradcam_metadata)} heatmap files to {heatmaps_dir}")

### 5.3) Save the metadata parquet and round-trip verify

The accumulated metadata rows become `gradcam_metadata.parquet`.
A round-trip verify confirms it loads cleanly — row count matches,
column list matches — and that a sample `.npy` file loads with
the expected (7, 7) shape. The two checks catch different failure
modes: the parquet round-trip catches a save corruption on the
index file, the `.npy` round-trip catches a save corruption on the
heatmap files themselves. Either failure would leave N04 unable
to read what this notebook produced; surfacing it here is far
better than two weeks later.

In [ ]:
### 5.3) Save gradcam_metadata.parquet and round-trip verify

# Save the metadata index.
metadata_path = OUTPUT_DIR / "gradcam_metadata.parquet"
gradcam_metadata.to_parquet(metadata_path)

# Round-trip verify the parquet — row count and columns match.
reloaded = pd.read_parquet(metadata_path)
assert len(reloaded) == len(gradcam_metadata), (
    f"Round-trip row count mismatch: wrote {len(gradcam_metadata)}, "
    f"read back {len(reloaded)}."
)
assert list(reloaded.columns) == list(gradcam_metadata.columns), (
    f"Round-trip column mismatch: wrote {list(gradcam_metadata.columns)}, "
    f"read back {list(reloaded.columns)}."
)

# Round-trip verify a sample .npy file. The metadata parquet stores
# heatmap_path as a relative path inside OUTPUT_DIR, so the load
# joins OUTPUT_DIR with the relative path.
sample_relpath = gradcam_metadata.iloc[0]["heatmap_path"]
sample_loaded = np.load(OUTPUT_DIR / sample_relpath)
assert sample_loaded.shape == (7, 7), (
    f"Sample heatmap has wrong shape: {sample_loaded.shape}, "
    f"expected (7, 7)."
)

print(f"Wrote: {metadata_path}")
print(f"  rows: {len(gradcam_metadata)}")
print(f"  columns: {list(gradcam_metadata.columns)}")
print(f"  sample .npy shape: {sample_loaded.shape}")
print(f"  round-trip verified: ✓")

## 6) Save the sanity-check results and point forward

Section 6 consolidates Sections 3, 4, and 5 into one JSON
deliverable: `sanity_check_results.json`. The file encodes
both verdicts (model-parameter randomization and data
randomization), the full trajectory data underlying each, the
methodology details a paper reader needs, and the methods-section
text the writeup will pull from.

The structure mirrors R2-Q1's JSON pattern:

- **`results`** — both verdicts plus the data behind them. The
  param-randomization block carries the 6-stage summary and the
  72-row per-probe trajectory; the data-randomization block
  carries the 12 per-probe ρ values plus the verdict and the
  `DEV_EPOCH_CAP` state at the time of the run.
- **`methodology`** — every methodological choice in one
  structured block: probe-set size and seed, target layer,
  randomization scheme, criterion, degeneracy handling, and the
  full methods-section text as a string field.
- **`interpretation`** — one paragraph summarizing what the
  verdicts mean for N04. Built from numeric variables so the
  wording reflects the actual run.

After the JSON lands, the section closes with a forward pointer
to N04 — what's on disk for it to consume and how the dependency
flows.

In [ ]:
### 6) Assemble sanity_check_results.json

import json

# Pre-compute the key headline numbers so the methods text and the JSON
# fields agree.
param_full_rho = float(
    stage_summary.query("stage == 'layer1'")["mean_rho"].iloc[0]
)
param_full_abs_rho = abs(param_full_rho)
data_abs_mean_rho = abs(data_random_mean_rho)
n_data_degenerate = sum(data_degenerate)

# Overall verdict: both checks must pass. If DEV_EPOCH_CAP gated the
# data-randomization verdict to "withheld", the overall verdict is also
# withheld — partial verification isn't a pass.
# Overall verdict: both checks must pass.
if not param_random_passes:
    overall_verdict = "fail"
elif data_random_verdict == "withheld":          # was: DEV_EPOCH_CAP is not None
    overall_verdict = "withheld"
elif data_random_verdict == "fail":              # was: not data_random_passes
    overall_verdict = "fail"
else:
    overall_verdict = "pass"

# Per-stage trajectory summary (6 rows, mirrors what 3.5 printed).
param_stage_summary = [
    {
        "stage": row["stage"],
        "mean_rho": float(row["mean_rho"]),
        "std_rho": float(row["std_rho"]),
        "n_degenerate": int(row["n_degenerate"]),
    }
    for _, row in stage_summary.iterrows()
]

# Per-probe long-format trajectory (72 rows, the full data behind the plot).
param_per_probe = [
    {
        "probe_idx": int(row["probe_idx"]),
        "stage": row["stage"],
        "rho": float(row["rho"]),
        "is_degenerate": bool(row["is_degenerate"]),
    }
    for _, row in trajectory.iterrows()
]

# Section 4 per-probe data (12 rows).
data_per_probe = [
    {
        "probe_idx": probe_idx,
        "rho": float(rho),
        "is_degenerate": bool(deg),
    }
    for probe_idx, (rho, deg) in enumerate(zip(data_rhos, data_degenerate))
]

# Per-stage methodology — uses STAGES from Section 3.4 directly so the
# JSON's stage list and the code's stage list cannot drift.
param_stage_methodology = [
    {"name": name, "randomizes": layers}
    for name, layers in STAGES
]

# Methods-section text — built from numeric variables so the wording
# reflects this run's actual values. The paper writeup will pull from
# this string field.
methods_text = (
    "Vanilla Grad-CAM (Selvaraju et al., 2017) was applied to the "
    "ResNet-18 classifier produced by R2-Q1, targeting the final "
    "convolutional block (model.layer4, a 7x7 feature map with 512 "
    "channels). Two sanity checks from Adebayo et al. (2018) were run "
    f"on a fixed {PROBE_SIZE}-image probe set drawn from the working set "
    f"with seed {PROBE_SEED}. "

    "The model-parameter randomization check cumulatively re-initialized "
    "the model's learnable parameters top-down across six stages "
    "(untouched; logits; layer4; layer3; layer2; layer1, conv1, and bn1), "
    "recomputing Grad-CAM at each stage and comparing to the trained-model "
    "baseline via Spearman rank correlation on the flattened 7x7 heatmap "
    "grids. The data-randomization check trained a separate model from "
    "scratch using R2-Q1's recipe (ResNet-18, SGD with momentum 0.9, "
    "lr 0.01->0.001 at epoch 7, batch size 32, 10 epochs) with class "
    "labels shuffled across the training split before the val carve, then "
    "compared its heatmaps to the trained-model baseline by the same "
    "metric. The Grad-CAM target class was held constant at the trained "
    "model's predicted class (pred_idx) across all comparisons, per "
    "Adebayo's adaptation. The pre-committed pass criterion was |mean "
    "Spearman rho| < 0.3 at full randomization. "

    f"The model-parameter check yielded |mean rho| = {param_full_abs_rho:.3f} "
    "at the deepest randomization stage (layer1 + initial conv); the "
    f"data-randomization check yielded |mean rho| = {data_abs_mean_rho:.3f}. "
    "Both fall well below the 0.3 threshold, supporting the use of the "
    "resulting heatmaps for downstream failure-mode categorization in "
    "Notebook 04. Heatmaps that were constant (all-zero after the ReLU) "
    "for a given probe were treated as ρ = 0 in the per-probe correlation; "
    f"this affected {n_data_degenerate}/{PROBE_SIZE} probes in the data-"
    "randomization check."
)

# Interpretation paragraph — branches on the overall verdict and is built
# from the numeric variables so the language reflects the run.
if overall_verdict == "pass":
    interpretation = (
        "Grad-CAM heatmaps from this classifier pass both Adebayo (2018) "
        "sanity checks. Downstream failure-mode interpretation in Notebook "
        "04 rests on a saliency method whose output is verifiably a "
        "function of what the trained model learned, not of input-data "
        "structure or saliency-method internals."
    )
elif overall_verdict == "withheld":
    interpretation = (
        "The model-parameter randomization check passed, but the data-"
        f"randomization check was run with DEV_EPOCH_CAP = {DEV_EPOCH_CAP} "
        "and is therefore withheld. The full 10-epoch shuffled retrain "
        "must be completed before the downstream failure-mode interpretation "
        "in Notebook 04 can be considered methodologically supported. "
        "Set DEV_EPOCH_CAP = None in Section 4 and delete "
        "shuffled_resnet18.pt to force a fresh full retrain."
    )
else:
    interpretation = (
        "Grad-CAM heatmaps from this classifier failed one or both Adebayo "
        "(2018) sanity checks. Downstream failure-mode interpretation in "
        "Notebook 04 cannot be supported by these heatmaps without "
        "additional methodological work — either a different saliency "
        "method that passes the checks, or documented limitations on what "
        "the categorization can claim."
    )

# Assemble the JSON.
sanity_check_results = {
    "results": {
        "overall_verdict": overall_verdict,
        "param_randomization": {
            "verdict": "pass" if param_random_passes else "fail",
            "verdict_mean_rho": param_full_rho,
            "verdict_abs_mean_rho": param_full_abs_rho,
            "stage_summary": param_stage_summary,
            "per_probe": param_per_probe,
        },
        "data_randomization": {
            "verdict": data_random_verdict,
            "verdict_mean_rho": float(data_random_mean_rho),
            "verdict_abs_mean_rho": data_abs_mean_rho,
            "n_degenerate": n_data_degenerate,
            "per_probe": data_per_probe,
            "dev_epoch_cap": DEV_EPOCH_CAP,
        },
    },
    "methodology": {
        "saliency_method": "vanilla Grad-CAM (Selvaraju et al. 2017)",
        "target_layer": "model.layer4",
        "target_class_policy": (
            "trained model's predicted class (pred_idx), held constant "
            "across all comparisons"
        ),
        "probe_set_size": PROBE_SIZE,
        "probe_set_seed": PROBE_SEED,
        "probe_set_source": (
            f"{PROBE_SIZE} rows sampled from working_set.parquet with "
            f"random_state={PROBE_SEED}"
        ),
        "criterion": "|mean Spearman rho| < 0.3 at full randomization",
        "param_randomization_stages": param_stage_methodology,
        "data_randomization_recipe": (
            "R2-Q1's training recipe (ResNet-18, SGD momentum 0.9, "
            "lr 0.01->0.001 at epoch 7, batch size 32, 10 epochs) via "
            "iri.train_baseline(shuffle_labels=True)"
        ),
        "degeneracy_handling": (
            "Heatmaps that are constant (all-zero after the ReLU) yield "
            "rho = 0 in the per-probe correlation; these probes are "
            "flagged is_degenerate=True in the per_probe arrays."
        ),
        "methods_text": methods_text,
    },
    "interpretation": interpretation,
}

# Save.
results_path = OUTPUT_DIR / "sanity_check_results.json"
with open(results_path, "w") as f:
    json.dump(sanity_check_results, f, indent=2)

# Round-trip verify: top-level keys, verdict fields, per-probe array
# lengths. Catches a JSON save corruption while not bloating the cell
# with a full deep equality check.
with open(results_path) as f:
    reloaded = json.load(f)

assert set(reloaded.keys()) == {"results", "methodology", "interpretation"}, (
    f"Top-level JSON keys mismatch: {set(reloaded.keys())}"
)
assert reloaded["results"]["overall_verdict"] == overall_verdict
assert len(reloaded["results"]["param_randomization"]["per_probe"]) == PROBE_SIZE * len(STAGES)
assert len(reloaded["results"]["data_randomization"]["per_probe"]) == PROBE_SIZE

print(f"Wrote: {results_path}")
print(f"  overall verdict:     {overall_verdict}")
print(f"  param-randomization: {reloaded['results']['param_randomization']['verdict']} "
      f"(|mean ρ| = {param_full_abs_rho:.3f})")
print(f"  data-randomization:  {reloaded['results']['data_randomization']['verdict']} "
      f"(|mean ρ| = {data_abs_mean_rho:.3f})")
print(f"  round-trip verified: ✓")

### Where to go from here

Week 3 is done. With both Adebayo sanity checks passed and the
Grad-CAM heatmaps generated for the full working set, the saliency
method underpinning R2-Q2 is verified and the data Week 4 needs
is on disk.

What this notebook produced, all in the R2-Q2 output directory:

- `sanity_check_trajectory.png` — the model-parameter randomization
  trajectory plot.
- `shuffled_resnet18.pt` — the label-shuffled-trained classifier
  used for the data-randomization check, cached so re-runs skip
  the 30-minute retrain.
- `heatmaps/` — 81 `.npy` files, one per working-set misclassification,
  each a 7×7 layer4 Grad-CAM heatmap pre-upsampling.
- `gradcam_metadata.parquet` — the index that maps working-set
  rows to .npy paths.
- `sanity_check_results.json` — both verdicts, full trajectory
  data, methodology, methods-section text, interpretation.

N04 (Week 4) picks up the failure-mode categorization. It reads
`gradcam_metadata.parquet` and the 81 `.npy` files, upsamples each
heatmap to its image's native resolution at use time, runs SAM-based
leaf segmentation, and applies the four numeric predicates committed
in N01 to assign each misclassification to a failure-mode category.
The output is `taxonomy_distribution.json` — the paper's headline
distribution.

The chain from R2-Q1 → R2-Q2 → paper is shallow and verifiable:
R2-Q1's classifier and its PD prediction table flow into R2-Q2's
working set; that working set flows into the heatmaps produced in
this notebook; those heatmaps flow into N04's categorization. Each
step's outputs are on disk and re-readable, and each step's
methodology is documented in its own JSON.